# K-Nearest Neighbors Code Companion

This notebook connects KNN code with the theory: distance, feature scaling, choosing K, classification, and evaluation.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)


## Load a Classification Dataset

KNN predicts a class by looking at the nearest training examples.


In [ ]:
iris = load_iris(as_frame=True)
X = iris.data
y = iris.target

print("Classes:", dict(enumerate(iris.target_names)))
X.head()


## Split the Data


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


## Train KNN with Feature Scaling

KNN depends on distance, so scaling is essential. Without scaling, large-number features can dominate the distance calculation.


In [ ]:
knn = Pipeline([
    ("scaler", StandardScaler()),
    ("model", KNeighborsClassifier(n_neighbors=5))
])

knn.fit(X_train, y_train)


## Evaluate on Training Folds


In [ ]:
scores = cross_val_score(knn, X_train, y_train, cv=cv, scoring="accuracy")
print("Mean CV accuracy:", scores.mean())
print("CV standard deviation:", scores.std())


## Compare Different K Values

Small K can overfit. Large K can underfit. Testing several K values helps us see the pattern.

Select parameters using five-fold cross-validation on training data only. The pipeline fits scaling separately inside each fold. GridSearchCV's `mean_test_score` refers to validation folds, not the held-out test set. After selection, the best pipeline is refitted on all training rows.

Fold standard deviations describe variability, not confidence intervals.


In [ ]:
search = GridSearchCV(
    knn,
    param_grid={"model__n_neighbors": list(range(1, 16))},
    scoring="accuracy",
    cv=cv,
    refit=True,
)
search.fit(X_train, y_train)
results = pd.DataFrame(search.cv_results_)
results["k"] = results["param_model__n_neighbors"].astype(int)
results["accuracy"] = results["mean_test_score"]
print("Selected parameters:", search.best_params_)
results[["params", "mean_test_score", "std_test_score", "rank_test_score"]]


In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(results["k"], results["accuracy"], marker="o")
plt.xlabel("K")
plt.ylabel("Mean cross-validation accuracy")
plt.title("Training-Fold Accuracy for Different K Values")
plt.xticks(results["k"])
plt.tight_layout()
plt.show()


## Final Held-Out Evaluation

Parameters are now fixed. Evaluate the selected model once on the untouched test set. Do not use this result to retune the model; that would turn the test set into validation data.


In [ ]:
y_pred = search.best_estimator_.predict(X_test)
print("Test accuracy:", accuracy_score(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred, target_names=iris.target_names))
